# Image Captioning with CNN-LSTM
This notebook demonstrates a complete pipeline for building an image captioning system using the Flickr8k dataset. We use a pre-trained **ResNet-50** as the encoder to extract image features and an **LSTM-based Decoder** to generate captions.

### 1. Setup & Configuration
First, we import the necessary libraries and define our hyperparameters.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
import os
import pandas as pd
from tqdm import tqdm
import re
from collections import Counter

# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
EMBED_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 1
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
NUM_EPOCHS = 5
VOCAB_MAX_SIZE = 10000
VOCAB_MIN_FREQ = 2

# Paths
ROOT_DIR = "dataset/raw/flickr8k/Images"
CAPTIONS_FILE = "dataset/raw/flickr8k/Flickr8k.token.txt"

Using device: cuda


### 2. Tokenizer and Dataset
We need a `Tokenizer` to convert text to numerical IDs and back. The `Flickr8kDataset` class handles loading images and captions. We've included logic to skip missing images to ensure robustness.

In [4]:
class Tokenizer:
    def __init__(self, max_vocab_size=10000, min_freq=2):
        self.max_vocab_size = max_vocab_size
        self.min_freq = min_freq
        self.specials = ["<PAD>", "<START>", "<END>", "<UNK>"]
        self.pad_idx = 0
        self.start_idx = 1
        self.end_idx = 2
        self.unk_idx = 3
        self.vocab = {t: i for i, t in enumerate(self.specials)}
        self.inv_vocab = {i: t for t, i in self.vocab.items()}

    def build_vocab(self, sentences):
        counter = Counter()
        for s in sentences:
            tokens = self._tokenize(s)
            counter.update(tokens)
        
        most_common = [w for w, c in counter.most_common(self.max_vocab_size) if c >= self.min_freq]
        for w in most_common:
            if w not in self.vocab:
                idx = len(self.vocab)
                self.vocab[w] = idx
        self.inv_vocab = {v: k for k, v in self.vocab.items()}

    def _tokenize(self, text):
        return re.sub(r"[^a-zA-Z0-9\s]", "", text.lower()).split()

    def encode(self, text, add_special=True):
        tokens = self._tokenize(text)
        ids = [self.vocab.get(t, self.unk_idx) for t in tokens]
        if add_special:
            ids = [self.start_idx] + ids + [self.end_idx]
        return ids

    def decode(self, ids):
        return " ".join([self.inv_vocab.get(i, "<UNK>") for i in ids if i not in [self.pad_idx, self.start_idx, self.end_idx]])

class Flickr8kDataset(Dataset):
    def __init__(self, root_dir, captions_file, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        raw_data = []
        with open(captions_file, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) < 2: continue
                img_id = parts[0].split('#')[0]
                caption = parts[1]
                raw_data.append((img_id, caption))
        
        # Validation
        print(f"Validating dataset images...")
        self.data = []
        existing_files = set(os.listdir(root_dir))
        for img_id, caption in raw_data:
            clean_id = img_id
            if '.jpg' in clean_id.lower() and not clean_id.lower().endswith('.jpg'):
                clean_id = clean_id[:clean_id.lower().find('.jpg') + 4]
            if clean_id in existing_files:
                self.data.append((clean_id, caption))
        print(f"Dataset Size: {len(self.data)}")

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_id, caption = self.data[idx]
        image = Image.open(os.path.join(self.root_dir, img_id)).convert("RGB")
        if self.transform: image = self.transform(image)
        return image, caption

### 3. Model Architecture
The model consists of two parts:
- **EncoderCNN**: A pre-trained ResNet-50 that extracts features from images.
- **DecoderRNN**: An LSTM that takes the image features and generates a sequence of words.
- **ImageCaptioner**: A wrapper that connects them.

In [5]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super(EncoderCNN, self).__init__()
        resnet = models.resnet50(pretrained=True)
        for param in resnet.parameters(): param.requires_grad = False
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.embed = nn.Linear(resnet.fc.in_features, embed_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):
        features = self.resnet(images)
        features = features.view(features.size(0), -1)
        return self.dropout(self.relu(self.embed(features)))

class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)

    def forward(self, features, captions):
        embeddings = self.embedding(captions[:, :-1])
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)
        hiddens, _ = self.lstm(embeddings)
        return self.linear(hiddens)

class ImageCaptioner(nn.Module):
    def __init__(self, encoder, decoder):
        super(ImageCaptioner, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):
        return self.decoder(self.encoder(images), captions)

### 4. Training
We initialize our components, define the loss function (CrossEntropy with padding ignored), and start the training loop.

In [ ]:
# 1. Prep Data
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

dataset = Flickr8kDataset(ROOT_DIR, CAPTIONS_FILE, transform=transform)
tokenizer = Tokenizer()
tokenizer.build_vocab([cap for _, cap in dataset.data])

def collate_fn(data):
    images, captions = zip(*data)
    images = torch.stack(images, 0)
    tokenized = [torch.tensor(tokenizer.encode(cap)) for cap in captions]
    padded = torch.nn.utils.rnn.pad_sequence(tokenized, batch_first=True, padding_value=tokenizer.pad_idx)
    return images, padded

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

# 2. Init Model
model = ImageCaptioner(
    EncoderCNN(EMBED_SIZE), 
    DecoderRNN(EMBED_SIZE, HIDDEN_SIZE, len(tokenizer.vocab), NUM_LAYERS)
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# 3. Training Loop
model.train()
for epoch in range(NUM_EPOCHS):
    loop = tqdm(loader, total=len(loader))
    for images, captions in loop:
        images, captions = images.to(device), captions.to(device)
        outputs = model(images, captions)
        loss = criterion(outputs.view(-1, len(tokenizer.vocab)), captions.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loop.set_description(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
        loop.set_postfix(loss=loss.item())

    # Save checkpoint
    torch.save(model.state_dict(), "model_checkpoint.pth")
    print(f"Epoch {epoch+1} completed.")

### 5. Inference
After training, we can use the model to generate captions for new images. We use **greedy search** to pick the most likely word at each step.

In [6]:
def generate_caption(image_path, model, tokenizer, max_len=20):
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    ])
    
    image = transform(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    
    with torch.no_grad():
        features = model.encoder(image) # [1, embed_size]
        states = None
        result_caption = []
        
        # Initial input (features)
        inputs = features.unsqueeze(1) # [1, 1, embed_size]
        
        for _ in range(max_len):
            hiddens, states = model.decoder.lstm(inputs, states)
            outputs = model.decoder.linear(hiddens.squeeze(1))
            predicted = outputs.argmax(1)
            
            idx = predicted.item()
            result_caption.append(idx)
            if idx == tokenizer.end_idx: break
            
            inputs = model.decoder.embedding(predicted).unsqueeze(1)
            
    return tokenizer.decode(result_caption)

# Test on an image from the dataset
test_img = os.path.join(ROOT_DIR, dataset.data[0][0])
print(f"Generated: {generate_caption(test_img, model, tokenizer)}")
print(f"Actual: {dataset.data[0][1]}")

NameError: name 'dataset' is not defined

### 6. Quantitative Evaluation (BLEU Scores)
To measure how well our model performs, we use the **BLEU (Bilingual Evaluation Understudy)** score. It compares the generated captions against several human-written reference captions.
- **BLEU-1**: Measures individual word overlap (unigrams).
- **BLEU-4**: Measures 4-word sequence overlap (quadgrams), indicating better grammatical structure.

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

def evaluate_bleu(model, test_ids, tokenizer, root_dir, captions_dict):
    model.eval()
    hypotheses = []
    references = []
    
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    ])

    for img_id in tqdm(test_ids):
        img_path = os.path.join(root_dir, img_id)
        if not os.path.exists(img_path): continue
            
        image = Image.open(img_path).convert("RGB")
        image = transform(image).unsqueeze(0).to(device)
        
        with torch.no_grad():
            features = model.encoder(image).unsqueeze(1)
            states = None
            prediction = []
            inputs = features
            for _ in range(20):
                hiddens, states = model.decoder.lstm(inputs, states)
                outputs = model.decoder.linear(hiddens.squeeze(1))
                predicted = outputs.argmax(1)
                idx = predicted.item()
                if idx == tokenizer.end_idx: break
                if idx not in [tokenizer.pad_idx, tokenizer.start_idx]:
                    prediction.append(tokenizer.inv_vocab.get(idx, "<UNK>"))
                inputs = model.decoder.embedding(predicted).unsqueeze(1)
            
            hypotheses.append(prediction)
            references.append(captions_dict[img_id])

    print(f"BLEU-1: {corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0)):.4f}")
    print(f"BLEU-4: {corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25)):.4f}")

# Map for evaluation
test_split_file = "dataset/raw/flickr8k/Flickr_8k.testImages.txt"
with open(test_split_file, 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

eval_refs = {}
with open(CAPTIONS_FILE, 'r') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) < 2: continue
        img = p[0].split('#')[0]
        if img in test_ids:
            if img not in eval_refs: eval_refs[img] = []
            eval_refs[img].append(tokenizer._tokenize(p[1]))

# Run evaluation (sampling first 100 for speed in notebook)
evaluate_bleu(model, test_ids[:100], tokenizer, ROOT_DIR, eval_refs)